In [11]:
import pandas as pd
import numpy as np

In [28]:
subgroup = pd.read_csv('idiopathic_subgroup.csv', index_col=0)
subgroup.columns = ['PATNO' , 'Group']

In [18]:
meta = pd.read_csv('./../../data/PPMI/MO-LLM/PPMI_META_LINKAGE.csv')

In [29]:
meta

,PATNO,ENROLL_DATE,ENROLL_AGE,CONCOHORT_DEFINITION,Subgroup,Genetic Status,GENDER,EVENT_ID,EDUCYRS,AGE_AT_VISIT,CONLRRK2,CONGBA,CONSNCA,BIRTHDT,race,ageonset,agediag,primdiag,changedx,Genetic_Mutation
0,40932,01/05/2014,62.0,Parkinson's Disease,Genetic,LRRK2,0.0,BL,18.0,62.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LRRK2
1,40989,01/05/2014,88.0,Parkinson's Disease,Genetic,LRRK2,0.0,BL,12.0,88.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LRRK2
2,41821,01/02/2017,87.0,Parkinson's Disease,Genetic,LRRK2,0.0,BL,14.0,87.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LRRK2
3,50770,01/02/2015,64.0,Prodromal,Genetic,LRRK2,0.0,BL,17.0,64.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LRRK2
4,50770,01/02/2015,64.0,Prodromal,Genetic,LRRK2,0.0,V06,17.0,66.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LRRK2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7975,203816,01/02/2023,71.9,Prodromal,NaN,NaN,1.0,BL,NaN,71.9,NaN,NaN,NaN,01/03/1951,NaN,NaN,NaN,NaN,NaN,NaN
7976,204012,01/03/2023,73.0,Parkinson's Disease,NaN,NaN,1.0,BL,NaN,73.0,NaN,NaN,NaN,01/03/1950,NaN,NaN,NaN,NaN,NaN,NaN
7977,205186,NaN,NaN,Parkinson's Disease,NaN,NaN,1.0,BL,NaN,71.2,NaN,NaN,NaN,01/12/1951,NaN,NaN,NaN,NaN,NaN,NaN
7978,211902,NaN,NaN,Parkinson's Disease,NaN,NaN,1.0,BL,NaN,64.6,NaN,NaN,NaN,01/07/1958,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
subgroup_meta = pd.merge(meta, subgroup , on='PATNO')[['PATNO','AGE_AT_VISIT' , 'EVENT_ID','GENDER' ,'Group']].drop_duplicates()

In [50]:
subgroup_meta

,PATNO,AGE_AT_VISIT,EVENT_ID,GENDER,Group
0,3001,65.1,BL,1.0,0
1,3001,65.6,V02,1.0,0
2,3001,66.2,V04,1.0,0
3,3001,67.3,V06,1.0,0
4,3001,68.3,V08,1.0,0
...,...,...,...,...,...
1962,150414,77.7,BL,1.0,2
1963,152582,73.3,BL,1.0,0
1964,152582,73.8,V02,1.0,0
1965,153194,77.0,BL,1.0,2


In [35]:
events = ['BL', 'V02' , 'V04' ,'V05',
          'V06' ,'V07' , 'V08' ,'V09' , 'V10' ,
          'V11' , 'V12' ,'V13' , 'V14' ,'V15' ,
          'V16' , 'V17' , 'V18','V19' ,'V20' ]

all_events = ['SC', 'BL', 'V01', 'V02', 'V03', 'V04', 'V05',
'V06', 'V07', 'V08', 'V09', 'V10', 'V11', 'V12',
'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19',
'ST', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25'
]

x_list = pd.DataFrame(np.arange(0 , 9.5 , 0.5) , index = events , columns=['response'])

In [56]:
events_df = pd.DataFrame()
for patno in subgroup_meta['PATNO'].unique() : 
    df_tmp = pd.DataFrame({'PATNO' : [patno for i in range(len(events))] , 'EVENT_ID' : events})
    events_df = pd.concat([events_df , df_tmp])

In [72]:
data = pd.merge(subgroup_meta , events_df , on=['PATNO' , 'EVENT_ID'], how='outer' )

data[['GENDER' , 'Group']] = data[['GENDER' , 'Group' ,'PATNO']].groupby('PATNO').transform(lambda x: x.fillna(x.mean()))

In [78]:

# Apply the function after grouping by 'Group'
data['AGE_AT_VISIT'] = data.groupby('Group')['AGE_AT_VISIT'].transform(custom_increment_fill)

In [76]:
# Function to incrementally fill NaN values within each group
def custom_increment_fill(group):
    last_value = None  # This will hold the last non-null value
    for i in range(len(group)):
        if pd.isna(group.iloc[i]):
            if last_value is not None:
                group.iloc[i] = last_value + 0.5
            # Update last_value only if current value is not NaN
            last_value = group.iloc[i]
        else:
            last_value = group.iloc[i]
    return group


In [80]:
data.to_csv('./subgroup_meta.csv')